LDA

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import pymorphy2
import re

In [3]:
# Путь к текстам и стоп-словам
corpus_path = Path('/Users/sergey/PycharmProjects/SemAnalyzer/Тематическое моделирование/100great_raw')
stopwords_path = Path('/Users/sergey/PycharmProjects/SemAnalyzer/Тематическое моделирование/swl.txt')

# Загрузка текстов
documents = []
for file in corpus_path.glob("*.txt"):
    with open(file, encoding='cp1251') as f:
        documents.append(f.read())

# Загрузка стоп-слов
with open(stopwords_path, encoding='utf-8') as f:
    stopwords = [line.strip() for line in f if line.strip()]

In [4]:
from natasha import Segmenter, MorphVocab, NewsEmbedding, NewsMorphTagger, Doc
import re

# Инициализация моделей Natasha (один раз)
segmenter = Segmenter()
morph_vocab = MorphVocab()
embedding = NewsEmbedding()
morph_tagger = NewsMorphTagger(embedding)

def tokenize_and_lemmatize_natasha(text):
    # Удаляем все, кроме русских букв и пробелов, приводим к нижнему регистру
    clean_text = re.sub(r'[^а-яё\s]', ' ', text.lower())
    doc = Doc(clean_text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)

    lemmas = []
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
        lemmas.append(token.lemma)
    return ' '.join(lemmas)

# Обработка всех документов
documents_processed = [tokenize_and_lemmatize_natasha(doc) for doc in documents]


In [5]:
print(documents_processed[0][:500])

спасибо что скачать книга в бесплатный электронный библиотека весь книга автор этот же книга в другой формат другой книга серия великий приятный чтение а ю низовский великий археологический открытие заря человечество человекообезьяна из южный африка и другой житель земля громкий заявление о тот что человек произойти от обезьяна прозвучать задолго до тот когда быть обнаружить первый реальный факт подтверждать или опровергать это утверждение история открытие останки ископаемый высокий примат гомин


In [6]:
def print_topics(model, feature_names, n_top_words):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_features = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        topics.append((f"Тема {topic_idx+1}", top_features))
    return pd.DataFrame(topics, columns=["Тема", f"Топ-{n_top_words} слов"])

In [7]:
# Параметры словаря
min_df = 5       # минимальная частота
max_df = 0.95    # максимальная доля документов

# Векторизация
vectorizer = CountVectorizer(stop_words=stopwords, max_df=max_df, min_df=min_df)
X = vectorizer.fit_transform(documents_processed)

# Слова словаря
feature_names = vectorizer.get_feature_names_out()

/Users/sergey/PycharmProjects/SemAnalyzer/.venvNew/lib/python3.9/site-packages/sklearn/feature_extraction/text.py:404: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ак', 'бляха', 'богу', 'бу', 'буль', 'вашему', 'вообще', 'всякому', 'го', 'другому', 'ей', 'елки', 'иному', 'кей', 'кис', 'кэй', 'ля', 'мое', 'моему', 'муха', 'нашему', 'нить', 'палки', 'паф', 'пиф', 'пли', 'своему', 'твоему', 'тик', 'тра', 'тс', 'ту', 'чегой', 'черт', 'чик', 'чтой', 'яй'] not in stop_words.
  warnings.warn(


In [8]:
# Число тем и количество слов на тему
n_topics = 10
n_top_words = 20

# Обучение LDA
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(X)

# Сохраняем результат
all_results = {}
key = f"{n_topics} тем / топ-{n_top_words}"
all_results[key] = print_topics(lda, feature_names, n_top_words)

In [9]:
all_results["10 тем / топ-20"]

,Тема,Топ-20 слов
0,Тема 1,"[двигатель, ток, самолет, скорость, колесо, ра..."
1,Тема 2,"[адмирал, эскадра, крейсер, порт, командование..."
2,Тема 3,"[архитектура, архитектор, архитектурный, фасад..."
3,Тема 4,"[научный, премия, физика, нобелевский, роман, ..."
4,Тема 5,"[фильм, петь, песня, певец, опера, певица, теа..."
5,Тема 6,"[фильм, актер, театр, актриса, спектакль, режи..."
6,Тема 7,"[археолог, раскопка, гробница, статуя, находка..."
7,Тема 8,"[орден, отряд, воин, медаль, командование, пол..."
8,Тема 9,"[заговор, правительство, заговорщик, переворот..."
9,Тема 10,"[живопись, полотно, творчество, пейзаж, композ..."


In [10]:
import pyLDAvis
import pyLDAvis.lda_model

# Подготовка данных для визуализации
vis_data = pyLDAvis.lda_model.prepare(lda, X, vectorizer)

# Включаем режим отображения в Jupyter Notebook
pyLDAvis.enable_notebook()

# Отображаем интерактивную визуализацию
pyLDAvis.display(vis_data)


pyLDAvis.save_html(vis_data, 'lda_10_topics.html')


In [11]:
print(f"Perplexity модели: {lda.perplexity(X):.2f}")

Perplexity модели: 5643.77


LSA

In [12]:
# Ячейка 1. Импортируем необходимые библиотеки
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD


In [13]:
# Ячейка 2. Функция для вывода тем — топ слов по каждой теме из модели LSA
def print_topics(svd_model, vectorizer, n_top_words):
    terms = vectorizer.get_feature_names_out()
    for i, comp in enumerate(svd_model.components_):
        terms_in_comp = zip(terms, comp)
        sorted_terms = sorted(terms_in_comp, key=lambda x: abs(x[1]), reverse=True)[:n_top_words]
        topic_words = [term for term, weight in sorted_terms]
        print(f"Тема {i+1}: {', '.join(topic_words)}")


In [14]:
min_df = 5
max_df = 0.95

# Преобразуем стоп-слова в список, если это множество
stopwords_list = list(stopwords) if not isinstance(stopwords, list) else stopwords

vectorizer = TfidfVectorizer(stop_words=stopwords_list, min_df=min_df, max_df=max_df)
X = vectorizer.fit_transform(documents_processed)

print(f"Матрица TF-IDF размером: {X.shape[0]} документов, {X.shape[1]} слов")

/Users/sergey/PycharmProjects/SemAnalyzer/.venvNew/lib/python3.9/site-packages/sklearn/feature_extraction/text.py:404: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ак', 'бляха', 'богу', 'бу', 'буль', 'вашему', 'вообще', 'всякому', 'го', 'другому', 'ей', 'елки', 'иному', 'кей', 'кис', 'кэй', 'ля', 'мое', 'моему', 'муха', 'нашему', 'нить', 'палки', 'паф', 'пиф', 'пли', 'своему', 'твоему', 'тик', 'тра', 'тс', 'ту', 'чегой', 'черт', 'чик', 'чтой', 'яй'] not in stop_words.
  warnings.warn(


Матрица TF-IDF размером: 22 документов, 17524 слов


In [15]:
import pandas as pd

n_topics = 10
n_top_words = 20

svd = TruncatedSVD(n_components=n_topics, random_state=42)
svd.fit(X)

terms = vectorizer.get_feature_names_out()

topics = []
for i, comp in enumerate(svd.components_):
    terms_in_comp = zip(terms, comp)
    sorted_terms = sorted(terms_in_comp, key=lambda x: abs(x[1]), reverse=True)[:n_top_words]
    topic_words = [term for term, weight in sorted_terms]
    topics.append({
        "Тема": f"Тема {i+1}",
        "Топ-20 слов": ", ".join(topic_words)
    })

topics_df = pd.DataFrame(topics)
display(topics_df)


,Тема,Топ-20 слов
0,Тема 1,"фильм, орден, живопись, театр, воин, любовь, т..."
1,Тема 2,"фильм, адмирал, актер, театр, отряд, командова..."
2,Тема 3,"фильм, актер, режиссер, актриса, театр, раскоп..."
3,Тема 4,"двигатель, ток, самолет, скорость, ракета, кол..."
4,Тема 5,"фильм, живопись, полотно, пейзаж, раскопка, тв..."
5,Тема 6,"фильм, архитектура, архитектор, научный, архит..."
6,Тема 7,"орден, орденский, медаль, премия, адмирал, эск..."
7,Тема 8,"орден, певица, архитектура, фильм, архитектор,..."
8,Тема 9,"певица, певец, орден, опера, петь, фильм, архи..."
9,Тема 10,"адмирал, эскадра, заговор, заговорщик, архитек..."


In [16]:
print("LDA Темы:")
display(all_results)

print("LSA Темы:")
display(topics_df)


LDA Темы:


{'10 тем / топ-20':       Тема                                        Топ-20 слов
 0   Тема 1  [двигатель, ток, самолет, скорость, колесо, ра...
 1   Тема 2  [адмирал, эскадра, крейсер, порт, командование...
 2   Тема 3  [архитектура, архитектор, архитектурный, фасад...
 3   Тема 4  [научный, премия, физика, нобелевский, роман, ...
 4   Тема 5  [фильм, петь, песня, певец, опера, певица, теа...
 5   Тема 6  [фильм, актер, театр, актриса, спектакль, режи...
 6   Тема 7  [археолог, раскопка, гробница, статуя, находка...
 7   Тема 8  [орден, отряд, воин, медаль, командование, пол...
 8   Тема 9  [заговор, правительство, заговорщик, переворот...
 9  Тема 10  [живопись, полотно, творчество, пейзаж, композ...}

LSA Темы:


,Тема,Топ-20 слов
0,Тема 1,"фильм, орден, живопись, театр, воин, любовь, т..."
1,Тема 2,"фильм, адмирал, актер, театр, отряд, командова..."
2,Тема 3,"фильм, актер, режиссер, актриса, театр, раскоп..."
3,Тема 4,"двигатель, ток, самолет, скорость, ракета, кол..."
4,Тема 5,"фильм, живопись, полотно, пейзаж, раскопка, тв..."
5,Тема 6,"фильм, архитектура, архитектор, научный, архит..."
6,Тема 7,"орден, орденский, медаль, премия, адмирал, эск..."
7,Тема 8,"орден, певица, архитектура, фильм, архитектор,..."
8,Тема 9,"певица, певец, орден, опера, петь, фильм, архи..."
9,Тема 10,"адмирал, эскадра, заговор, заговорщик, архитек..."


BERTopic

In [17]:
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP


/Users/sergey/PycharmProjects/SemAnalyzer/.venvNew/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning:

urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020



In [19]:
from hdbscan import HDBSCAN
hdbscan_model = HDBSCAN(min_cluster_size=2, metric='euclidean', prediction_data=True)

In [20]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from sklearn.pipeline import make_pipeline

# Настройка параметров векторизации
vectorizer_model = CountVectorizer(
    min_df = 1,
    max_df = 19,
    stop_words=stopwords
)

# Настройка модели уменьшения размерности
umap_model = UMAP(n_neighbors=5, n_components=15, min_dist=0.0, metric='cosine', random_state=42)

# Создание и обучение модели BERTopic
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=None,
    language="multilingual",  # Учитывая, что тексты на русском
    verbose=True
)

# Обучение модели на ваших лемматизированных данных
topics, probs = topic_model.fit_transform(documents_processed)

# Выводим топики
topic_model.get_topic_info()

2025-06-19 17:51:25,624 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-06-19 17:51:44,210 - BERTopic - Embedding - Completed ✓
2025-06-19 17:51:44,211 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2025-06-19 17:51:46,877 - BERTopic - Dimensionality - Completed ✓
2025-06-19 17:51:46,878 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-06-19 17:51:46,889 - BERTopic - Cluster - Completed ✓
2025-06-19 17:51:46,896 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-06-19 17:51:48,215 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2,-1_бог_человек_ток_год,"[бог, человек, ток, год, машина, иметь, время,...",[первый бог как возникнуть у человек идея бог ...
1,0,11,0_год_время_стать_картина,"[год, время, стать, картина, человек, художник...",[спасибо что скачать книга в бесплатный электр...
2,1,5,1_год_город_время_орден,"[год, город, время, орден, стать, век, древний...",[спасибо что скачать книга в бесплатный электр...
3,2,4,2_год_роль_жизнь_театр,"[год, роль, жизнь, театр, фильм, человек, стат...",[спасибо что скачать книга в бесплатный электр...
